In [414]:
import pandas as pd
import os
import json
import numpy as np

In [415]:
# Define the directory containing your CSV and JSON files
csv_directory = "../src/data_to_reDate/csv"
json_directory = "../src/data_to_reDate/json"

In [416]:
# Read CSV Files
csv_files = os.listdir(csv_directory)
csv_dataframes = []
for file in csv_files:
    if file.endswith(".csv"):
        file_path = os.path.join(csv_directory, file)
        df = pd.read_csv(file_path)
        csv_dataframes.append(df)

In [417]:
# Read JSON Files
json_files = os.listdir(json_directory)
json_dataframes = []
for file in json_files:
    if file.endswith(".json"):
        file_path = os.path.join(json_directory, file)
        df = pd.read_json(file_path)
        json_dataframes.append(df)

In [418]:
# Merge DataFrames
merged_csv = pd.concat(csv_dataframes, ignore_index=True)
merged_json = pd.concat(json_dataframes, ignore_index=True)

In [419]:
def extract_data(row):
    return {
        "startTime": row["startTime"],
        "cloudBase": row["values"]["cloudBase"],
        "cloudCeiling": row["values"]["cloudCeiling"],
        "cloudCover": row["values"]["cloudCover"],
        "dewPoint": row["values"]["dewPoint"],
        "freezingRainIntensity": row["values"]["freezingRainIntensity"],
        "humidity": row["values"]["humidity"],
        "precipitationProbability": row["values"]["precipitationProbability"],
        "pressureSurfaceLevel": row["values"]["pressureSurfaceLevel"],
        "rainIntensity": row["values"]["rainIntensity"],
        "sleetIntensity": row["values"]["sleetIntensity"],
        "snowIntensity": row["values"]["snowIntensity"],
        "temperature": row["values"]["temperature"],
        "temperatureApparent": row["values"]["temperatureApparent"],
        "uvHealthConcern": row["values"]["uvHealthConcern"],
        "uvIndex": row["values"]["uvIndex"],
        "visibility": row["values"]["visibility"],
        "weatherCode": row["values"]["weatherCode"],
        "windDirection": row["values"]["windDirection"],
        "windGust": row["values"]["windGust"],
        "windSpeed": row["values"]["windSpeed"],
    }

In [420]:
def validate_json(json_str):
    try:
        json_obj = json.loads(json_str)
        return json_obj
    except json.decoder.JSONDecodeError as e:
        # Print the problematic part of the JSON string
        error_location = e.pos
        print(
            "Problematic JSON part:",
            json_str[max(0, error_location - 10) : error_location + 10],
        )
        print("Invalid JSON:", e)
        return None

In [421]:
def preprocess_json_string(json_str):
    # Replace occurrences of "None" as a string with None
    return json_str.replace("'", '"').replace("None", "null")

In [422]:
def process_intervals(value):
    # List to store individual DataFrames
    dataframes = []

    # Iterate over each row in the JSON data
    if type(value) == str:
        value = validate_json(preprocess_json_string(value))
    for entry in value:
        # Extract relevant data from the row
        data = extract_data(entry)

        # Create DataFrame from extracted data
        df = pd.DataFrame([data])

        # Append DataFrame to list
        dataframes.append(df)

    # Concatenate all DataFrames into a single DataFrame
    final_df = pd.concat(dataframes, ignore_index=True)

    return final_df

In [423]:
# Example usage:
result_dfs_csv = []

# Iterate over each row in the merged_dataframe
for index, row in merged_csv.iterrows():
    # Process intervals column for each row
    final_df = process_intervals(row["intervals"])
    # Append the result DataFrame to the list
    result_dfs_csv.append(final_df)

# Concatenate all DataFrames into a single DataFrame
final_result_df_csv = pd.concat(result_dfs_csv, ignore_index=True)

# Display the final result DataFrame
final_result_df_csv

/tmp/ipykernel_23398/3513322562.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(dataframes, ignore_index=True)
/tmp/ipykernel_23398/3104010016.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_result_df_csv = pd.concat(result_dfs_csv, ignore_index=True)


,startTime,cloudBase,cloudCeiling,cloudCover,dewPoint,freezingRainIntensity,humidity,precipitationProbability,pressureSurfaceLevel,rainIntensity,...,snowIntensity,temperature,temperatureApparent,uvHealthConcern,uvIndex,visibility,weatherCode,windDirection,windGust,windSpeed
0,2024-03-13T00:14:00+07:00,1.17,NaN,44.00,20.00,0,58.00,0,1012.51,0.0,...,0,29.00,30.73,0,0,9.99,1101,150.00,6.14,4.63
1,2024-03-13T00:29:00+07:00,1.17,NaN,44.00,20.00,0,58.00,0,1012.40,0.0,...,0,29.00,30.73,0,0,9.99,1101,150.00,5.89,4.63
2,2024-03-13T00:44:00+07:00,0.55,0.55,96.27,22.16,0,76.67,0,1012.29,0.0,...,0,26.67,26.67,0,0,15.60,1001,152.88,5.70,3.69
3,2024-03-13T00:59:00+07:00,1.02,NaN,47.73,20.15,0,63.07,0,1012.18,0.0,...,0,27.90,29.64,0,0,10.39,1101,140.00,5.51,3.63
4,2024-03-13T01:14:00+07:00,1.02,NaN,44.00,20.00,0,62.00,0,1012.04,0.0,...,0,28.00,29.68,0,0,9.99,1101,140.00,4.74,3.63
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2129,2024-03-04T23:01:00+07:00,0.39,NaN,44.00,24.00,0,84.00,0,1011.84,0.0,...,0,27.00,30.12,0,0,9.99,1101,170.00,6.80,3.13
2130,2024-03-04T23:16:00+07:00,0.39,NaN,44.00,24.00,0,84.00,0,1011.83,0.0,...,0,27.00,30.12,0,0,9.99,1101,150.00,6.68,3.13
2131,2024-03-04T23:31:00+07:00,0.39,NaN,44.00,24.00,0,84.00,0,1011.82,0.0,...,0,27.00,30.12,0,0,9.99,1101,150.00,6.62,3.13
2132,2024-03-04T23:46:00+07:00,0.39,NaN,44.00,24.00,0,84.00,0,1011.82,0.0,...,0,27.00,30.12,0,0,9.99,1101,150.00,6.50,3.13


In [424]:
# Example usage:
result_dfs_json = []

# Iterate over each row in the merged_dataframe
for index, row in merged_json.iterrows():
    # Process intervals column for each row
    final_df = process_intervals((row["intervals"]))
    # Append the result DataFrame to the list
    result_dfs_json.append(final_df)

# Concatenate all DataFrames into a single DataFrame
final_result_df_json = pd.concat(result_dfs_json, ignore_index=True)

# Display the final result DataFrame
final_result_df_json

/tmp/ipykernel_23398/3513322562.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(dataframes, ignore_index=True)


,startTime,cloudBase,cloudCeiling,cloudCover,dewPoint,freezingRainIntensity,humidity,precipitationProbability,pressureSurfaceLevel,rainIntensity,...,snowIntensity,temperature,temperatureApparent,uvHealthConcern,uvIndex,visibility,weatherCode,windDirection,windGust,windSpeed
0,2024-03-22T00:07:00+07:00,0.77,NaN,44.0,22.0,0,70.0,5,1010.85,0.15,...,0,28.0,30.67,0,0,9.99,1101,150.0,5.85,4.13
1,2024-03-22T00:22:00+07:00,0.77,NaN,44.0,22.0,0,70.0,5,1010.67,0.25,...,0,28.0,30.67,0,0,9.99,1101,150.0,5.75,3.89
2,2024-03-22T00:37:00+07:00,0.77,NaN,44.0,22.0,0,70.0,5,1010.48,0.13,...,0,28.0,30.67,0,0,9.99,1101,130.0,5.66,3.63
3,2024-03-22T00:52:00+07:00,0.77,NaN,44.0,22.0,0,70.0,5,1010.30,1.27,...,0,28.0,30.67,0,0,9.99,4200,130.0,5.57,4.09
4,2024-03-22T01:07:00+07:00,0.77,NaN,44.0,22.0,0,70.0,5,1010.15,0.13,...,0,28.0,30.67,0,0,9.99,1101,140.0,5.56,4.63
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,2024-03-21T23:01:00+07:00,0.77,NaN,44.0,22.0,0,70.0,0,1011.97,0.00,...,0,28.0,30.67,0,0,9.99,1101,140.0,6.49,3.66
287,2024-03-21T23:16:00+07:00,0.77,NaN,44.0,22.0,0,70.0,0,1011.85,0.00,...,0,28.0,30.67,0,0,9.99,1101,120.0,6.36,4.13
288,2024-03-21T23:31:00+07:00,0.77,NaN,44.0,22.0,0,70.0,0,1011.73,0.00,...,0,28.0,30.67,0,0,9.99,1101,120.0,6.18,4.13
289,2024-03-21T23:46:00+07:00,0.77,NaN,44.0,22.0,0,70.0,0,1011.61,0.00,...,0,28.0,30.67,0,0,9.99,1101,120.0,6.13,4.13


In [425]:
# # Optionally, you can merge the CSV and JSON DataFrames together if they have the same columns
merged_dataframe = pd.concat(
    [final_result_df_csv, final_result_df_json], ignore_index=True
)

# # Display the merged DataFrame
merged_dataframe

,startTime,cloudBase,cloudCeiling,cloudCover,dewPoint,freezingRainIntensity,humidity,precipitationProbability,pressureSurfaceLevel,rainIntensity,...,snowIntensity,temperature,temperatureApparent,uvHealthConcern,uvIndex,visibility,weatherCode,windDirection,windGust,windSpeed
0,2024-03-13T00:14:00+07:00,1.17,NaN,44.00,20.00,0,58.00,0,1012.51,0.00,...,0,29.00,30.73,0,0,9.99,1101,150.00,6.14,4.63
1,2024-03-13T00:29:00+07:00,1.17,NaN,44.00,20.00,0,58.00,0,1012.40,0.00,...,0,29.00,30.73,0,0,9.99,1101,150.00,5.89,4.63
2,2024-03-13T00:44:00+07:00,0.55,0.55,96.27,22.16,0,76.67,0,1012.29,0.00,...,0,26.67,26.67,0,0,15.60,1001,152.88,5.70,3.69
3,2024-03-13T00:59:00+07:00,1.02,NaN,47.73,20.15,0,63.07,0,1012.18,0.00,...,0,27.90,29.64,0,0,10.39,1101,140.00,5.51,3.63
4,2024-03-13T01:14:00+07:00,1.02,NaN,44.00,20.00,0,62.00,0,1012.04,0.00,...,0,28.00,29.68,0,0,9.99,1101,140.00,4.74,3.63
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2420,2024-03-21T23:01:00+07:00,0.77,NaN,44.00,22.00,0,70.00,0,1011.97,0.00,...,0,28.00,30.67,0,0,9.99,1101,140.00,6.49,3.66
2421,2024-03-21T23:16:00+07:00,0.77,NaN,44.00,22.00,0,70.00,0,1011.85,0.00,...,0,28.00,30.67,0,0,9.99,1101,120.00,6.36,4.13
2422,2024-03-21T23:31:00+07:00,0.77,NaN,44.00,22.00,0,70.00,0,1011.73,0.00,...,0,28.00,30.67,0,0,9.99,1101,120.00,6.18,4.13
2423,2024-03-21T23:46:00+07:00,0.77,NaN,44.00,22.00,0,70.00,0,1011.61,0.00,...,0,28.00,30.67,0,0,9.99,1101,120.00,6.13,4.13


In [426]:
# Convert 'startTime' to datetime and handle errors
try:
    merged_dataframe["Time"] = pd.to_datetime(
        merged_dataframe["startTime"], errors="raise"
    )
except ValueError as e:
    print("Error converting some values to datetime:", e)

# Sort the DataFrame based on 'Time'
merged_dataframe = merged_dataframe.sort_values(by="Time")

# Arrange columns
cols = list(merged_dataframe.columns)
cols = ["Time"] + [col for col in cols if col != "Time"]
merged_dataframe = merged_dataframe[cols]

# Drop column and Reset index
merged_dataframe.drop(columns=["startTime"], inplace=True)
merged_dataframe.reset_index(drop=True, inplace=True)

merged_dataframe

,Time,cloudBase,cloudCeiling,cloudCover,dewPoint,freezingRainIntensity,humidity,precipitationProbability,pressureSurfaceLevel,rainIntensity,...,snowIntensity,temperature,temperatureApparent,uvHealthConcern,uvIndex,visibility,weatherCode,windDirection,windGust,windSpeed
0,2024-02-20 23:54:00+07:00,0.33,NaN,21.2,21.31,0,85.60,0,1006.98,0.0,...,0,23.84,23.84,0,0,15.46,1100,152.63,7.76,4.58
1,2024-02-21 00:09:00+07:00,0.32,NaN,20.6,21.31,0,86.00,0,1006.88,0.0,...,0,23.74,23.74,0,0,15.24,1100,153.38,7.81,4.63
2,2024-02-21 00:24:00+07:00,0.31,NaN,22.8,21.31,0,86.60,0,1006.73,0.0,...,0,23.69,23.69,0,0,14.88,1100,153.50,7.74,4.55
3,2024-02-21 00:39:00+07:00,0.30,NaN,24.6,21.31,0,87.00,0,1006.59,0.0,...,0,23.65,23.65,0,0,14.52,1100,154.38,7.65,4.50
4,2024-02-21 00:54:00+07:00,0.28,NaN,26.2,21.31,0,87.60,0,1006.44,0.0,...,0,23.55,23.55,0,0,14.17,1100,155.31,7.63,4.43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2420,2024-03-22 23:07:00+07:00,0.52,NaN,44.0,24.00,0,79.00,0,1010.96,0.0,...,0,28.00,31.95,0,0,9.99,1101,150.00,6.23,2.63
2421,2024-03-22 23:22:00+07:00,0.52,NaN,44.0,24.00,0,79.00,0,1010.88,0.0,...,0,28.00,31.95,0,0,9.99,1101,150.00,6.07,2.86
2422,2024-03-22 23:37:00+07:00,0.52,NaN,44.0,24.00,0,79.00,0,1010.79,0.0,...,0,28.00,31.95,0,0,9.99,1101,140.00,5.91,3.13
2423,2024-03-22 23:52:00+07:00,0.52,NaN,44.0,24.00,0,81.33,0,1010.71,0.0,...,0,27.53,31.13,0,0,9.99,1101,140.00,5.75,3.13


In [427]:
# Find the minimum date
min_date = merged_dataframe["Time"].min()

# Calculate the number of days to shift
shift_days = (
    pd.Timestamp("2024-02-28").tz_localize("Asia/Ho_Chi_Minh") - min_date
).days

# Shift the 'Time' column
merged_dataframe["Time"] += pd.Timedelta(days=shift_days)

merged_dataframe

,Time,cloudBase,cloudCeiling,cloudCover,dewPoint,freezingRainIntensity,humidity,precipitationProbability,pressureSurfaceLevel,rainIntensity,...,snowIntensity,temperature,temperatureApparent,uvHealthConcern,uvIndex,visibility,weatherCode,windDirection,windGust,windSpeed
0,2024-02-27 23:54:00+07:00,0.33,NaN,21.2,21.31,0,85.60,0,1006.98,0.0,...,0,23.84,23.84,0,0,15.46,1100,152.63,7.76,4.58
1,2024-02-28 00:09:00+07:00,0.32,NaN,20.6,21.31,0,86.00,0,1006.88,0.0,...,0,23.74,23.74,0,0,15.24,1100,153.38,7.81,4.63
2,2024-02-28 00:24:00+07:00,0.31,NaN,22.8,21.31,0,86.60,0,1006.73,0.0,...,0,23.69,23.69,0,0,14.88,1100,153.50,7.74,4.55
3,2024-02-28 00:39:00+07:00,0.30,NaN,24.6,21.31,0,87.00,0,1006.59,0.0,...,0,23.65,23.65,0,0,14.52,1100,154.38,7.65,4.50
4,2024-02-28 00:54:00+07:00,0.28,NaN,26.2,21.31,0,87.60,0,1006.44,0.0,...,0,23.55,23.55,0,0,14.17,1100,155.31,7.63,4.43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2420,2024-03-29 23:07:00+07:00,0.52,NaN,44.0,24.00,0,79.00,0,1010.96,0.0,...,0,28.00,31.95,0,0,9.99,1101,150.00,6.23,2.63
2421,2024-03-29 23:22:00+07:00,0.52,NaN,44.0,24.00,0,79.00,0,1010.88,0.0,...,0,28.00,31.95,0,0,9.99,1101,150.00,6.07,2.86
2422,2024-03-29 23:37:00+07:00,0.52,NaN,44.0,24.00,0,79.00,0,1010.79,0.0,...,0,28.00,31.95,0,0,9.99,1101,140.00,5.91,3.13
2423,2024-03-29 23:52:00+07:00,0.52,NaN,44.0,24.00,0,81.33,0,1010.71,0.0,...,0,27.53,31.13,0,0,9.99,1101,140.00,5.75,3.13


In [428]:
merged_dataframe.to_csv("../src/data_to_reDate/merge_date_data.csv")